# HeartBench Exploratory Analysis

This notebook provides exploratory analysis of pilot and main results for both benchmarks:
- **Moral Stories Subset** (primary, 240 items from Emelin et al., 2021)
- **HeartBench-240** (secondary, 240 custom items)

In [ ]:
import json
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
print(f'Project root: {PROJECT_ROOT}')

## 1. Load Benchmarks

In [ ]:
from utils import load_benchmark, load_jsonl

# Primary: Moral Stories subset
ms_path = PROJECT_ROOT / 'benchmark' / 'moral_stories_subset.jsonl'
if ms_path.exists():
    ms_benchmark = load_benchmark(ms_path)
    df_ms = pd.DataFrame(ms_benchmark)
    print(f'Moral Stories subset: {len(df_ms)} items')
    display(df_ms.head())
else:
    print('Moral Stories subset not found. Run: python3 src/build_moral_stories_subset.py')

# Secondary: HeartBench-240
hb_path = PROJECT_ROOT / 'benchmark' / 'heartbench_240.jsonl'
if hb_path.exists():
    hb_benchmark = load_benchmark(hb_path)
    df_hb = pd.DataFrame(hb_benchmark)
    print(f'\nHeartBench-240: {len(df_hb)} items')
else:
    print('HeartBench-240 not found. Run: python3 src/build_benchmark.py')

In [ ]:
# Distribution summaries
print('Family distribution:')
print(df_bench['family'].value_counts())
print('\nDifficulty distribution:')
print(df_bench['difficulty'].value_counts())
print('\nGold heart_worse balance:')
print(df_bench['gold_heart_worse'].value_counts())

## 2. Load Pilot Results (Moral Stories - Primary)

In [ ]:
# Try new-style path first, then legacy path
ms_pilot_path = PROJECT_ROOT / 'results' / 'moral_stories' / 'pilot' / 'all_results.jsonl'
hb_pilot_path = PROJECT_ROOT / 'results' / 'pilot' / 'all_results.jsonl'

for label, pilot_path, bench in [
    ('Moral Stories', ms_pilot_path, ms_benchmark if ms_path.exists() else []),
    ('HeartBench', hb_pilot_path, hb_benchmark if hb_path.exists() else []),
]:
    if not pilot_path.exists():
        print(f'{label}: No pilot results found at {pilot_path}')
        continue
    pilot_results = load_jsonl(pilot_path)
    df_pilot = pd.DataFrame(pilot_results)
    print(f'\n=== {label} Pilot ===')
    print(f'Results: {len(df_pilot)}')
    print(f'Models: {df_pilot["model"].unique()}')
    print(f'Conditions: {df_pilot["condition"].unique()}')
    print(f'Parse success rate: {df_pilot["parse_success"].mean():.3f}')
    
    # Merge with gold labels
    gold_map = {item['item_id']: item for item in bench}
    df_pilot['gold_heart_worse'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_heart_worse'))
    df_pilot['gold_morally_worse'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_morally_worse'))
    df_pilot['family'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('family'))
    df_pilot['heart_correct'] = df_pilot.apply(lambda r: r['heart_worse'] == r['gold_heart_worse'] if r['parse_success'] else None, axis=1)
    df_pilot['moral_correct'] = df_pilot.apply(lambda r: r['morally_worse'] == r['gold_morally_worse'] if r['parse_success'] else None, axis=1)
    
    valid = df_pilot[df_pilot['parse_success'] == True]
    summary = valid.groupby(['model', 'condition']).agg(
        heart_acc=('heart_correct', 'mean'),
        moral_acc=('moral_correct', 'mean'),
        n=('heart_correct', 'count')
    ).round(3)
    print(f'\n{summary}')

In [ ]:
# Merge with gold labels
if pilot_path.exists():
    gold_map = {item['item_id']: item for item in benchmark}
    df_pilot['gold_heart_worse'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_heart_worse'))
    df_pilot['gold_morally_worse'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_morally_worse'))
    df_pilot['family'] = df_pilot['item_id'].map(lambda x: gold_map.get(x, {}).get('family'))
    df_pilot['heart_correct'] = df_pilot.apply(lambda r: r['heart_worse'] == r['gold_heart_worse'] if r['parse_success'] else None, axis=1)
    df_pilot['moral_correct'] = df_pilot.apply(lambda r: r['morally_worse'] == r['gold_morally_worse'] if r['parse_success'] else None, axis=1)

## 3. Heart Accuracy by Condition and Model

In [ ]:
if pilot_path.exists():
    valid = df_pilot[df_pilot['parse_success'] == True]
    summary = valid.groupby(['model', 'condition']).agg(
        heart_acc=('heart_correct', 'mean'),
        moral_acc=('moral_correct', 'mean'),
        n=('heart_correct', 'count')
    ).round(3)
    print(summary)

## 4. Reason Focus Distribution

In [ ]:
if pilot_path.exists():
    valid_rf = df_pilot[df_pilot['parse_success'] == True]
    rf_table = pd.crosstab(valid_rf['condition'], valid_rf['reason_focus'], normalize='index').round(3)
    print('Reason focus distribution by condition:')
    print(rf_table)

## 5. Per-Family Accuracy

In [ ]:
if pilot_path.exists():
    family_summary = valid.groupby(['model', 'condition', 'family']).agg(
        heart_acc=('heart_correct', 'mean'),
        n=('heart_correct', 'count')
    ).round(3)
    print(family_summary)

## 6. Load Scores and Analysis (if available)

In [ ]:
scores_path = PROJECT_ROOT / 'results' / 'pilot' / 'scores.json'
if scores_path.exists():
    with open(scores_path) as f:
        scores = json.load(f)
    for key, s in scores.get('per_model_condition', {}).items():
        ha = s['heart_accuracy']
        print(f"{key}: heart_acc={ha['accuracy']:.3f} (n={ha['n']})")
else:
    print('No scores found. Run: python3 src/score_results.py')

analysis_path = PROJECT_ROOT / 'results' / 'pilot' / 'analysis.json'
if analysis_path.exists():
    with open(analysis_path) as f:
        analysis = json.load(f)
    print('\nPaired analysis loaded.')
else:
    print('No analysis found. Run: python3 src/analyze_results.py')

## 7. Full Run Results — Position Bias Analysis

The critical finding from full evaluation: prompt framing primarily shifts **position bias** (which letter the model defaults to), not moral reasoning ability.

In [ ]:
# Load full run results for both benchmarks
from utils import load_jsonl

benchmarks_full = {}
for bench_name, bench_data, results_dir in [
    ('moral_stories', ms_benchmark if ms_path.exists() else [], 'moral_stories/main'),
    ('heartbench', hb_benchmark if hb_path.exists() else [], 'heartbench/main'),
]:
    rpath = PROJECT_ROOT / 'results' / results_dir / 'all_results.jsonl'
    if not rpath.exists():
        print(f'{bench_name}: No full results at {rpath}')
        continue
    results = load_jsonl(rpath)
    df = pd.DataFrame(results)
    gold_map = {item['item_id']: item for item in bench_data}
    df['gold_heart_worse'] = df['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_heart_worse'))
    df['gold_morally_worse'] = df['item_id'].map(lambda x: gold_map.get(x, {}).get('gold_morally_worse'))
    df['family'] = df['item_id'].map(lambda x: gold_map.get(x, {}).get('family'))
    df['heart_correct'] = df.apply(
        lambda r: r.get('heart_worse') == r['gold_heart_worse'] if r.get('parse_success') else None, axis=1)
    benchmarks_full[bench_name] = df
    models = df['model'].unique()
    print(f'{bench_name}: {len(df)} results, models: {list(models)}')
    
    # Summary table
    valid = df[df['parse_success'] == True]
    summary = valid.groupby(['model', 'condition']).agg(
        heart_acc=('heart_correct', 'mean'),
        n=('heart_correct', 'count')
    ).round(3)
    print(summary)
    print()

In [ ]:
# Position Bias Analysis: % of answers that are "A" by model x condition
print('=== POSITION BIAS: % Answers "A" ===\n')

for bench_name, df in benchmarks_full.items():
    print(f'--- {bench_name} ---')
    valid = df[df['parse_success'] == True].copy()
    valid['is_A'] = (valid['heart_worse'] == 'A').astype(float)
    
    pivot = valid.groupby(['model', 'condition'])['is_A'].agg(['mean', 'count']).round(3)
    pivot.columns = ['pct_A', 'n']
    print(pivot)
    print()

    # Position-debiased accuracy: accuracy conditioned on gold label
    print(f'--- {bench_name}: Conditional Accuracy ---')
    for model in sorted(valid['model'].unique()):
        for cond in sorted(valid['condition'].unique()):
            subset = valid[(valid['model'] == model) & (valid['condition'] == cond)]
            gold_a = subset[subset['gold_heart_worse'] == 'A']
            gold_b = subset[subset['gold_heart_worse'] == 'B']
            acc_a = gold_a['heart_correct'].mean() if len(gold_a) > 0 else float('nan')
            acc_b = gold_b['heart_correct'].mean() if len(gold_b) > 0 else float('nan')
            balanced = (acc_a + acc_b) / 2
            print(f'  {model}/{cond}: acc|gold=A={acc_a:.3f}, acc|gold=B={acc_b:.3f}, balanced={balanced:.3f}')
    print()

## 8. Moral-Heart Dissociation

How often does heart_worse differ from morally_worse? High dissociation under a framing condition suggests the model treats these as separate judgments — but doesn't necessarily improve accuracy.

In [ ]:
# Moral-Heart Dissociation Rate by condition
print('=== MORAL-HEART DISSOCIATION RATE ===\n')

for bench_name, df in benchmarks_full.items():
    print(f'--- {bench_name} ---')
    valid = df[(df['parse_success'] == True)].copy()
    valid['dissociated'] = valid.apply(
        lambda r: r.get('heart_worse') != r.get('morally_worse') 
        if r.get('heart_worse') and r.get('morally_worse') else None, axis=1)
    
    dissoc = valid.dropna(subset=['dissociated']).groupby(['model', 'condition'])['dissociated'].mean().round(3)
    print(dissoc)
    print()